In [ ]:
import os
import sys

os.chdir("..")
sys.path.append(os.getcwd())

import pandas as pd
import torch
from ensemble import *
from datasets import get_dataloaders
from models import get_model
from sklearn.metrics import accuracy_score

In [ ]:
device = "cuda"

_, _, test_loader = get_dataloaders(128)

In [ ]:
# load the models we want to ensemble
paths = [
    "../results/exp2_aug/model_0_smallcnn_0.pth",
    "../results/exp2_aug/model_1_bigcnn_0.pth",
]

models = []

for p in paths:
    m = get_model("smallcnn", 0.3) 
    m.load_state_dict(torch.load(p))
    m.to(device)
    models.append(m)

In [ ]:
probs = collect_outputs(models, test_loader, device)

In [ ]:
soft_preds = soft_voting_cached(probs)
hard_preds = hard_voting_cached(probs)

weights = [0.6, 0.4]
weighted_preds = weighted_soft_voting_cached(probs, weights)

In [ ]:
labels = []
for _, y in test_loader:
    labels.extend(y.numpy())

soft_acc = accuracy_score(labels, soft_preds)
hard_acc = accuracy_score(labels, hard_preds)
weighted_acc = accuracy_score(labels, weighted_preds)

print("Soft:", soft_acc)
print("Hard:", hard_acc)
print("Weighted:", weighted_acc)

In [ ]:


results = pd.DataFrame({
    "method": ["soft", "hard", "weighted"],
    "accuracy": [soft_acc, hard_acc, weighted_acc]
})

results